In [ ]:
!pip -q install iterative-stratification

In [ ]:
import json
import pandas as pd
from pathlib import Path

from google.colab import drive
from google.colab import userdata
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
import joblib
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


In [ ]:
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/thesis_results/SCOTBESS_splits")
INPUT_DIR = Path("/content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2")
INPUT_PATH = INPUT_DIR / "SCOTBESS_FULL_ANNOTATED_TERRA_LOW.csv"

df = pd.read_csv(INPUT_PATH)
df["label_list"] = df["labels"].apply(json.loads)

print(f"Responses: {len(df):,}")
print(f"Projects: {df['project'].nunique()}")
print(f"Labels: {len({label for labels in df['label_list'] for label in labels})}")

Mounted at /content/drive
Responses: 1,675
Projects: 91
Labels: 20


## Label distribution inspection

In [ ]:
project_sizes = df.groupby("project").size().sort_values(ascending=False).rename("responses").reset_index()

display(project_sizes)

,project,responses
0,East_Renfrewshire_2024_0168_TP,212
1,Aberdeenshire_APP_2025_0415,196
2,Aberdeenshire_APP_2024_2125,115
3,Scottish_Government_ECU00005160,94
4,Aberdeen_City_240614_DPP,89
...,...,...
86,Scottish_Government_ECU00004631,1
87,Scottish_Government_ECU00004979,1
88,Scottish_Government_ECU00004991,1
89,Scottish_Government_ECU00006107,1


In [ ]:
display(project_sizes["responses"].describe())

,responses
count,91.000000
mean,18.406593
std,35.897068
min,1.000000
25%,2.000000
50%,5.000000
75%,13.500000
max,212.000000


In [ ]:
df["label_list"] = df["labels"].apply(json.loads)

label_counts = df["label_list"].explode().dropna().value_counts().rename("responses").reset_index()
label_counts.columns = ["label", "responses"]
label_counts["percentage"] = 100 * label_counts["responses"] / len(df)

display(label_counts)

,label,responses,percentage
0,Fire and Explosion Risk,910,54.328358
1,"Landscape, Visual and Heritage Impact",885,52.835821
2,Site Selection,838,50.029851
3,Traffic,709,42.328358
4,Wildlife and Ecology,690,41.194030
5,Noise,660,39.402985
6,Health and Wellbeing,636,37.970149
7,Residential Proximity and Separation Distance,610,36.417910
8,Emergency Planning and Response,515,30.746269
9,Water and Soil Contamination,484,28.895522


In [ ]:
exploded = df[["document_id", "project", "label_list"]].explode("label_list").dropna(subset=["label_list"])

label_project = exploded.groupby(["label_list", "project"]).size().unstack(fill_value=0)

coverage = pd.DataFrame({
    "total_responses": label_project.sum(axis=1),
    "projects_with_label": (label_project > 0).sum(axis=1),
    "max_in_single_project": label_project.max(axis=1)})

coverage["largest_project_share"] = coverage["max_in_single_project"] / coverage["total_responses"]
coverage = coverage.sort_values("projects_with_label")

display(coverage)

,total_responses,projects_with_label,max_in_single_project,largest_project_share
label_list,,,,
Decommissioning and Site Restoration,190,34,38,0.200000
Property Value,148,38,19,0.128378
Light Pollution,269,44,56,0.208178
Agricultural Land,311,44,46,0.147910
Grid Connection and Electrical Infrastructure,174,45,21,0.120690
Community and Economic Benefits,294,50,33,0.112245
Water and Soil Contamination,484,55,81,0.167355
Planning Policy and Regulatory Compliance,453,57,69,0.152318
Project Need,323,58,39,0.120743


In [ ]:
empty_labels = df["label_list"].apply(len).eq(0)

print(f"No-label responses: {empty_labels.sum()}/{len(df)} ({100 * empty_labels.mean():.2f}%)")

No-label responses: 35/1675 (2.09%)


In [ ]:
empty_by_project = df.assign(no_label=empty_labels).groupby("project")["no_label"].agg(["sum", "count"])
empty_by_project["percentage"] = 100 * empty_by_project["sum"] / empty_by_project["count"]

display(empty_by_project.sort_values("sum", ascending=False))

,sum,count,percentage
project,,,
Scottish_Government_ECU00006053,6,50,12.000000
Aberdeenshire_APP_2025_0415,5,196,2.551020
Aberdeen_City_240614_DPP,3,89,3.370787
Aberdeenshire_APP_2025_1015,2,54,3.703704
East_Renfrewshire_2024_0168_TP,2,212,0.943396
...,...,...,...
Scottish_Government_ECU00006111,0,7,0.000000
Scottish_Government_ECU00006121,0,24,0.000000
Scottish_Government_ECU00006204,0,5,0.000000


## Iterative multi-label stratified split (train/validation/test)




In [ ]:
label_list = sorted({label for labels in df["label_list"] for label in labels})

mlb = MultiLabelBinarizer(classes=label_list)
Y = mlb.fit_transform(df["label_list"])

print(mlb.classes_)
print(Y.shape)

MLB_PATH = PROJECT_DIR / "scotbess_mlb.joblib"
joblib.dump(mlb, MLB_PATH, compress=3)

print(f"Saved: {MLB_PATH}")

['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']
(1675, 20)
Saved: /content/drive/MyDrive/thesis_results/SCOTBESS_splits/scotbess_mlb.joblib


In [ ]:
SEED = 42
X_dummy = np.zeros((len(df), 1))

#80% train / 20% temporary
split_1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED)

train_idx, temp_idx = next(split_1.split(X_dummy, Y))

#split remaining 20% into ~10% validation and ~10% test
split_2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=SEED)

val_rel_idx, test_rel_idx = next(split_2.split(X_dummy[temp_idx], Y[temp_idx]))

test_idx = temp_idx[val_rel_idx]
val_idx = temp_idx[test_rel_idx]

print(f"Train: {len(train_idx)} ({len(train_idx)/len(df):.2%})")
print(f"Validation: {len(val_idx)} ({len(val_idx)/len(df):.2%})")
print(f"Test: {len(test_idx)} ({len(test_idx)/len(df):.2%})")

Train: 1340 (80.00%)
Validation: 165 (9.85%)
Test: 170 (10.15%)


In [ ]:
df["split"] = ""


df.loc[train_idx, "split"] = "train"
df.loc[val_idx, "split"] = "validation"
df.loc[test_idx, "split"] = "test"

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()



print(df["split"].value_counts())

split
train         1340
test           170
validation     165
Name: count, dtype: int64


In [ ]:
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert len(train_idx) + len(val_idx) + len(test_idx) == len(df)

print("Split integrity check passed.")

Split integrity check passed.


In [ ]:
split_label_counts = pd.DataFrame(index=label_list)

for name, subset in {
    "train": train_df,
    "validation": val_df,
    "test": test_df}.items():
    split_label_counts[name] = [subset["label_list"].apply(lambda x: label in x).sum() for label in label_list]

display(split_label_counts)

assert (split_label_counts > 0).all().all()
print("All labels occur in all three splits.")

,train,validation,test
Agricultural Land,249,31,31
Community and Economic Benefits,235,30,29
"Consultation, Transparency and Information",363,46,45
Cumulative Impact,381,48,47
Decommissioning and Site Restoration,152,19,19
Emergency Planning and Response,412,52,51
Fire and Explosion Risk,728,91,91
Grid Connection and Electrical Infrastructure,139,18,17
Health and Wellbeing,509,64,63
"Landscape, Visual and Heritage Impact",702,92,91


All labels occur in all three splits.


In [ ]:
#how common a particular label is in the split
split_prevalence = pd.DataFrame(index=label_list)

split_prevalence["full"] = [df["label_list"].apply(lambda x: label in x).mean() for label in label_list]

for name, subset in {
    "train": train_df,
    "validation": val_df,
    "test": test_df}.items():
    split_prevalence[name] = [subset["label_list"].apply(lambda x: label in x).mean() for label in label_list]

display((split_prevalence * 100).round(2))

,full,train,validation,test
Agricultural Land,18.57,18.58,18.79,18.24
Community and Economic Benefits,17.55,17.54,18.18,17.06
"Consultation, Transparency and Information",27.10,27.09,27.88,26.47
Cumulative Impact,28.42,28.43,29.09,27.65
Decommissioning and Site Restoration,11.34,11.34,11.52,11.18
Emergency Planning and Response,30.75,30.75,31.52,30.00
Fire and Explosion Risk,54.33,54.33,55.15,53.53
Grid Connection and Electrical Infrastructure,10.39,10.37,10.91,10.00
Health and Wellbeing,37.97,37.99,38.79,37.06
"Landscape, Visual and Heritage Impact",52.84,52.39,55.76,53.53


In [ ]:
SPLIT_DATA_PATH = PROJECT_DIR / "SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv"
df.drop(columns=["label_list"]).to_csv(SPLIT_DATA_PATH, index=False)


print(f"Saved: {SPLIT_DATA_PATH}")

Saved: /content/drive/MyDrive/thesis_results/SCOTBESS_splits/SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv
